# `aidp-fusion-bicc` live test — BICC extract → Object Storage → Spark CSV
**Live-test row 14.** End-to-end: trigger BICC, wait, read CSV from OCI Object Storage.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.auth import http_basic_session
from oracle_ai_data_platform_connectors.rest.fusion import (
    trigger_bicc_extract, read_bicc_csv_from_object_storage,
)

session = http_basic_session(
    username=os.environ['FUSION_BICC_USER'],
    password=os.environ['FUSION_BICC_PASSWORD'],
    base_url=os.environ['FUSION_BICC_BASE_URL'],
)
prefix = trigger_bicc_extract(
    session=session,
    base_url=os.environ['FUSION_BICC_BASE_URL'],
    offering=os.environ['FUSION_BICC_OFFERING'],
    poll_interval_seconds=30, timeout_seconds=3600,
)
print('extract prefix:', prefix)


In [ ]:
df = read_bicc_csv_from_object_storage(
    spark=spark,
    namespace=os.environ['OCI_NAMESPACE'],
    bucket=os.environ['OCI_BUCKET_BICC'],
    prefix=prefix,
)
df.printSchema()


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-fusion-bicc',
    'auth': 'basic-plus-apikey',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
